In [1]:
from datetime import datetime
from dotenv import load_dotenv
from dateutil.relativedelta import relativedelta  #is calender aware, as months are fixed No. days
from newsapi import NewsApiClient
import pandas as pd
import csv
import os


load_dotenv()
token = os.getenv('newsapi_TOKEN')
newsapi = NewsApiClient(api_key=token)

In [2]:
def get_date_pairs():
    current_date = datetime.today().date()
    start_date = current_date - relativedelta(months=1)

    date_pairs = []

    while start_date<current_date:
        next_day_date = start_date + relativedelta(days=1)
        date_pairs.append((start_date,next_day_date))
        start_date = next_day_date

    return date_pairs

In [3]:
def get_news_per_day():
    news_data = []
    for i in get_date_pairs():
        all_top_headlines = newsapi.get_everything(q='(oil OR crude OR energy OR brent OR wti) AND (iran OR price OR prices OR supply OR demand OR production OR disruption OR volatility OR surge OR drop)',
                                           from_param=i[0],
                                           to=i[1],
                                           language='en',
                                           sort_by='relevancy')
        articles = all_top_headlines['articles']
        news_data.extend(articles)
    return news_data
    
news_data = get_news_per_day()

In [4]:
normalised_dataframe = pd.DataFrame(pd.json_normalize(news_data))
normalised_dataframe.drop_duplicates(subset=['title'],inplace=True)

def relevancy(row):
    text_to_run = (str(row['title']) + ' ' + str(row['description'])).lower()
    needed = ['oil','brent','wti','fuel','energy','opec','iran','strikes']
    return any(i in text_to_run for i in needed)

normalised_dataframe['relevant'] = normalised_dataframe.apply(relevancy, axis=1)
normalised_dataframe = normalised_dataframe[normalised_dataframe['relevant']==True]


In [5]:
if os.path.exists('NewsAPI_data_multipleday.csv'):
    normalised_dataframe.to_csv('NewsAPI_data_multipleday.csv', index=False, mode='a', header=False, encoding='utf-8')
    print('Dataset updated')

else:
    normalised_dataframe.to_csv('NewsAPI_data_multipleday.csv', index=False, encoding='utf-8')
    print('File does not exist, creating new')

Dataset updated


In [7]:
normalised_dataframe

,author,title,description,url,urlToImage,publishedAt,content,source.id,source.name,relevant
0,Lauren Edmonds,The US secretary of energy says Iran is not a ...,"US Energy Secretary Chris Wright blamed ""emoti...",https://www.businessinsider.com/gas-prices-oil...,https://i.insider.com/69ad9e3cd3e2f1aef36a32d4...,2026-03-08T18:35:45Z,US Energy Secretary Chris Wright.Rebecca Black...,business-insider,Business Insider,True
1,ABC News,WATCH: War in Iran's impact on gas,"Oil prices continue to soar, rattling stock ma...",https://abcnews.com/video/130858678/,https://i.abcnewsfe.com/a/d43a1ddf-e38c-40d0-9...,2026-03-07T16:29:23Z,"<ul><li>US says it has hit 3,000 targets in Ir...",NaN,Abcnews.com,True
2,Al Jazeera,Iran war threatens prolonged impact on energy ...,The conflict has already led to the suspension...,https://www.aljazeera.com/news/2026/3/8/iran-w...,https://www.aljazeera.com/wp-content/uploads/2...,2026-03-08T12:55:57Z,The United States-Israeli war on Iran could le...,al-jazeera-english,Al Jazeera English,True
3,John Power,Iran war is latest threat to a global economy ...,Rising energy prices threaten to stoke inflati...,https://www.aljazeera.com/economy/2026/3/7/ira...,https://www.aljazeera.com/wp-content/uploads/2...,2026-03-07T00:27:18Z,As the United States and Israels war on Iran u...,al-jazeera-english,Al Jazeera English,True
4,Alex Bitter,Gen Z is adopting cats at higher rates than do...,Pet-care app Rover is seeing more demand for c...,https://www.businessinsider.com/gen-z-loves-ca...,https://i.insider.com/69aafe86d3e2f1aef36a2107...,2026-03-08T09:30:01Z,"Rover is seeing an uptick in cat ownership, pa...",business-insider,Business Insider,True
...,...,...,...,...,...,...,...,...,...,...
2970,Lance D Johnson,Last-ditch 45-day truce talks fail as Iran rej...,"Mediators from Pakistan, Egypt and Turkey push...",https://www.naturalnews.com/2026-04-06-last-di...,https://www.naturalnews.com/wp-content/uploads...,2026-04-06T06:00:00Z,"<ul><li>Mediators from Pakistan, Egypt and Tur...",NaN,Naturalnews.com,True
2972,RTÉ News,Trump vows 'hell' for Iran over Strait but wan...,US President Donald Trump has threatened to ra...,https://www.rte.ie/news/world/2026/0406/156687...,https://www.rte.ie/images/00242bca-1600.jpg,2026-04-06T04:00:02Z,US President Donald Trump has threatened to ra...,rte,RTE,True
2975,Swastika Das Sharma,AirAsia X to increase fares as jet fuel prices...,AirAsia X co-founder Tony Fernandes said in a ...,https://www.livemint.com/companies/news/airasi...,https://www.livemint.com/lm-img/img/2026/04/06...,2026-04-06T06:11:30Z,Budget carrier AirAsia X on Monday said it was...,NaN,Livemint,True
2976,Sputnik International,Iran Tells IAEA Shelling of Bushehr NPP Is War...,TEHRAN (Sputnik) - The shelling of the Bushehr...,https://sputnikglobe.com/20260406/iran-tells-i...,https://cdn1.img.sputnikglobe.com/images/shari...,2026-04-06T08:35:56Z,The Israeli and US military launched Operation...,NaN,Sputnikglobe.com,True
